## **Gdańsk vehicle feed**

In [3]:
import requests

url = "https://ckan2.multimediagdansk.pl/gpsPositions?v=2"
response = requests.get(url, timeout=20)
print(response.status_code)

200


In [7]:
data = response.json()
print(data.keys())
print(data["lastUpdate"])
print(len(data["vehicles"]))
print(data["vehicles"][0])

dict_keys(['lastUpdate', 'vehicles'])
2026-11-11T10:52:39Z
184
{'generated': '2026-11-11T10:52:35Z', 'routeShortName': '155', 'tripId': 92, 'routeId': 155, 'headsign': 'Łostowice Świętokrzyska', 'vehicleCode': '2812', 'vehicleService': '155-03', 'vehicleId': 145991, 'speed': 30, 'direction': 45, 'delay': 6, 'scheduledTripStartTime': '2026-11-11T10:28:00Z', 'lat': 54.31858825683594, 'lon': 18.58909034729004, 'gpsQuality': 3}


Fetched 11 Nov 11:52 Warsaw time: 184 vehicles, JSON, no API key needed.

- `lastUpdate` - time of the whole snapshot (UTC)
- `vehicles` - list of vehicles:
    - `generated` - when the position was recorded (UTC)
    - `routeShortName` - line number
    - `tripId` + `scheduledTripStartTime` - identify the scheduled run
    - `headsign` - destination
    - `vehicleCode` - number on the vehicle
    - `delay` - probably seconds?
    - `lat`, `lon` - position
    - `speed`, `direction`, `gpsQuality`, `routeId`, `vehicleId`, `vehicleService` - not explored yet

In [9]:
import time
from datetime import datetime, timezone

for i in range(3):
    data = requests.get(url, timeout=20).json()
    now = datetime.now(timezone.utc)
    ages = [
        (now - datetime.fromisoformat(v["generated"].replace("Z", "+00:00"))).total_seconds()
        for v in data["vehicles"]
    ]
    print(
        f"lastUpdate={data['lastUpdate']} vehicles={len(ages)}"
        f"newest={min(ages):.0f}s oldest={max(ages):.0f}s"
    )
    snapshot_age = now - datetime.fromisoformat(data["lastUpdate"].replace("Z", "+00:00"))
    print(snapshot_age.total_seconds())
    time.sleep(30)

lastUpdate=2026-11-11T11:17:49Z vehicles=196
newest=105s oldest=190s
104.92497
lastUpdate=2026-11-11T11:18:09Z vehicles=196
newest=115s oldest=220s
115.256716
lastUpdate=2026-11-11T11:18:52Z vehicles=198
newest=102s oldest=250s
102.457953


Log refreshes every ~20–40s.  \
Lags are around ~15-25s.  \
Some vehicles stop updating (stale positions should be filtered).    \

## **Gdańsk trip updates feed**

In [11]:
from google.transit import gtfs_realtime_pb2

tu_url = "https://ckan2.multimediagdansk.pl/gtfs-rt?feed=tripUpdates"
feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(requests.get(tu_url, timeout=20).content)

# print(feed.header)
print(feed)
# print(len(feed.entity))
# print(feed.entity[0])

header {
  gtfs_realtime_version: "2.0"
  incrementality: FULL_DATASET
  timestamp: 1794396714
}
entity {
  id: "127202611111215_202_127-08"
  trip_update {
    trip {
      trip_id: "127202611111215_202_127-08"
      schedule_relationship: SCHEDULED
    }
    stop_time_update {
      stop_sequence: 11
      arrival {
        delay: 156
      }
      stop_id: "1368"
      schedule_relationship: SCHEDULED
    }
    stop_time_update {
      stop_sequence: 12
      arrival {
        delay: 135
      }
      stop_id: "1366"
      schedule_relationship: SCHEDULED
    }
    stop_time_update {
      stop_sequence: 13
      arrival {
        delay: 115
      }
      stop_id: "1364"
      schedule_relationship: SCHEDULED
    }
    stop_time_update {
      stop_sequence: 14
      arrival {
        delay: 64
      }
      stop_id: "1362"
      schedule_relationship: SCHEDULED
    }
    stop_time_update {
      stop_sequence: 15
      arrival {
        delay: 0
      }
      stop_id: "1594"
      

Protobuf, 259 trips at 12:27, no API key, delay per upcoming stop

- `header`:
    - `gtfs_realtime_version` - version
    - `incrementality` - "FULL_DATASET" means each fetch is a complete snapshot, not just the changes
    - `timestamp` - meaning seconds since 1970 in UTC
- list of entities:
    - `entity` - a specific trip:
        - `id` - trip_id with date, start time and route?
        - `trip_update` - a list of updates:
            - `trip` info about the trip:
                - `trip_id` - same as id of the entity
                - `schedule_relationship` - "SCHEDULED", "ADDED", "CANCELED" and so on
            - `stop_time_update` - info about each stop:
                - `stop_sequence` - position of the stop along this trip (8 = 8th stop)
                - `arrival.delay` - predicted delay in seconds
                - `stop_id`, `schedule_relationship` - self explanatory

In [12]:
v = data["vehicles"][0]
print(v["routeId"], v["tripId"], v["scheduledTripStartTime"], v["vehicleService"])

matches = [
    e.id for e in feed.entity if e.id.startswith(str(v["routeId"])) and f"_{v['tripId']}_" in e.id
]
print(matches)

106 101 2026-11-11T10:51:00Z 106-22
['106202611111151_101_106-22']


Feeds join: trip update ID = routeId + date + local start time (HHMM) + \_tripId\_ + vehicleService. \
Checked on one vehicle; verify on more later.

## Gdynia feed - delays per stop

In [16]:
gd_url = "https://api.zdiz.gdynia.pl/pt/delays"
r = requests.get(gd_url, params={"stopId": 37030}, timeout=20)
print(r.status_code)
g = r.json()
print(g.keys())
print(g["lastUpdate"])
print(len(g["delay"]))
print(g["delay"])

200
dict_keys(['lastUpdate', 'delay'])
2026-11-11 13:02:11
4
[{'id': 'T41R10026', 'delayInSeconds': 83, 'estimatedTime': '13:01', 'headsign': 'Cisowa Sibeliusa 03', 'routeId': 10026, 'tripId': 41, 'status': 'REALTIME', 'theoreticalTime': '13:00', 'timestamp': '13:00:35', 'trip': 5898939, 'vehicleCode': 3068, 'vehicleId': 1486}, {'id': 'T31R10141', 'delayInSeconds': 38, 'estimatedTime': '13:01', 'headsign': 'Pogórze Górne 01', 'routeId': 10141, 'tripId': 31, 'status': 'REALTIME', 'theoreticalTime': '13:01', 'timestamp': '13:00:39', 'trip': 5900835, 'vehicleCode': 5001, 'vehicleId': 1206}, {'id': 'T41R10026', 'delayInSeconds': -142, 'estimatedTime': '13:27', 'headsign': 'Cisowa Sibeliusa 03', 'routeId': 10026, 'tripId': 41, 'status': 'REALTIME', 'theoreticalTime': '13:30', 'timestamp': '13:00:37', 'trip': 5899162, 'vehicleCode': 3072, 'vehicleId': 1490}, {'id': 'T31R10141', 'delayInSeconds': -64, 'estimatedTime': '13:29', 'headsign': 'Pogórze Górne 01', 'routeId': 10141, 'tripId': 31, 's

Fetched 11 Nov 12:53 Warsaw time, stop 37030: 2 departures, JSON, no API key.
One request per stop (~1,100 stop poles), no citywide snapshot

- `lastUpdate` is local time with no timezone
- a list with the delays:
    - `headsign` - final destination
    - `routeId` - id of the route (can be mapped with `/pt/routes`)
    - `trip`, `tripId` - two trip IDs, meaning unclear
    - `theoreticalTime`, `estimatedTime` - are HH:MM with no date
    - `delayInSeconds` - can be negative (early)
    - `status` REALTIME = live prediction
    - `id`, `status`, `timestamp` (local time), `vehicleId`, `vehicleCode` - self-explanatory

Other endpoints (found by testing, not documented): `/pt/routes`, `/pt/trips`, `/pt/stops`, `/pt/agency`, `/pt/stop_times` (38 MB+, daily at most).

## Decision
- Live data:
    - Poller collects: Gdańsk trip updates (main source, delay per stop) + Gdańsk vehicle positions, every ~30 s
    - Gdynia (`/pt/delay`): poll a chosen list of key stops (to be picked), not all 1,100
    - Fallback: if Gdynia breaks, MVP runs on Gdańsk only
- Timetable data (collect daily):
    - `routes`, `trips` and `stops` to turn IDs into line numbers, names and locations
    - `stop_times` to compare against the schedule

- SKM: still a stretch goal, not explored